In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from normal_evaluation.drbart_evaluation import *

In [2]:
model_name = 'bpic_2017_all_2'
#model_name = 'bpic_2017_all'
log_name = 'test'
with open('../transformed_event_logs/BPIC_2017_all_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)

test_event_log['case:concept:name'] = test_event_log['case:concept:name'].astype(str)
known_resources = ['User_1','User_10','User_100','User_101','User_102','User_103','User_104','User_105','User_106','User_107','User_108','User_109','User_11','User_110','User_111','User_112','User_113','User_114','User_115','User_116','User_117','User_118','User_119','User_12','User_120','User_121','User_122','User_123','User_124','User_125','User_126','User_127','User_128','User_129','User_13','User_130','User_131','User_132','User_133','User_134','User_135','User_136','User_137','User_138','User_139','User_14','User_140','User_141','User_142','User_143','User_144','User_145','User_146','User_147','User_148','User_149','User_15','User_16','User_17','User_18','User_19','User_2','User_20','User_21','User_22','User_23','User_24','User_25','User_26','User_27','User_28','User_29','User_3','User_30','User_31','User_32','User_33','User_34','User_35','User_36','User_37','User_38','User_39','User_4','User_40','User_41','User_42','User_43','User_44','User_45','User_46','User_47','User_48','User_49','User_5','User_50','User_51','User_52','User_53','User_54','User_55','User_56','User_57','User_58','User_59','User_6','User_60','User_61','User_62','User_63','User_64','User_65','User_66','User_67','User_68','User_69','User_7','User_70','User_71','User_72','User_73','User_74','User_75','User_76','User_77','User_78','User_79','User_8','User_80','User_81','User_82','User_83','User_84','User_85','User_86','User_87','User_88','User_89','User_9','User_90','User_91','User_92','User_93','User_94','User_95','User_96','User_97','User_98','User_99']
known_activities = ['W_Assess potential fraud__ate_abort','W_Assess potential fraud__complete','W_Assess potential fraud__resume','W_Assess potential fraud__schedule','W_Assess potential fraud__start','W_Assess potential fraud__suspend','W_Assess potential fraud__withdraw','W_Call after offers__ate_abort','W_Call after offers__complete','W_Call after offers__resume','W_Call after offers__schedule','W_Call after offers__start','W_Call after offers__suspend','W_Call after offers__withdraw','W_Call incomplete files__ate_abort','W_Call incomplete files__complete','W_Call incomplete files__resume','W_Call incomplete files__schedule','W_Call incomplete files__start','W_Call incomplete files__suspend','W_Complete application__ate_abort','W_Complete application__complete','W_Complete application__resume','W_Complete application__schedule','W_Complete application__start','W_Complete application__suspend','W_Handle leads__complete','W_Handle leads__resume','W_Handle leads__schedule','W_Handle leads__start','W_Handle leads__suspend','W_Handle leads__withdraw','W_Shortened completion __resume','W_Shortened completion __schedule','W_Shortened completion __start','W_Shortened completion __suspend','W_Validate application__ate_abort','W_Validate application__complete','W_Validate application__resume','W_Validate application__schedule','W_Validate application__start','W_Validate application__suspend']

/tmp/ipykernel_2141592/1655077238.py:5: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  test_event_log = pickle.load(f)


In [3]:
N = 1000
import conduct_evaluation
get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

likelihoods_A = None
likelihoods_R = None
likelihoods_R_A = None
likelihoods_R_A_S = None
likelihoods_R_A_S_AC = None
likelihoods_R_A_S_RC = None
likelihoods_R_A_S_RC_AC = None
likelihoods_R_A_S_RC_AC_V = None
likelihoods_R_A_S_D = None
likelihoods_R_A_S_D_RC_AC = None

In [4]:
drbart_model_A = DRBART(parser_dir = '../../../models/'+model_name+'/concept-name/',
                     strict_parser=False)
evaluator_A = conduct_evaluation.ConductEvaluation(drbart_model_A, SampleOutcomes_DRBART_Normal_A, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                    },
                                     test_event_log, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

FileNotFoundError: [Errno 2] No such file or directory: '../../../models/bpic_2017_all_2/concept-name/ucuts.json'

In [5]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

TypeError: 'NoneType' object is not subscriptable

In [6]:
np.mean(get_pscores(likelihoods_A))

TypeError: 'NoneType' object is not subscriptable

In [7]:
drbart_model_R_A = DRBART(parser_dir = '../../../models/'+model_name+'/concept-name_resource/',
                     strict_parser=False)
evaluator_R_A = conduct_evaluation.ConductEvaluation(drbart_model_R_A, SampleOutcomes_DRBART_Normal_R_A,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A = evaluator_R_A.sample_cases(False, True)

FileNotFoundError: [Errno 2] No such file or directory: '../../../models/bpic_2017_all_2/concept-name_resource/ucuts.json'

In [8]:
np.mean([v.ln() for v in likelihoods_R_A[0].values()])

TypeError: 'NoneType' object is not subscriptable

In [9]:
np.mean(get_pscores(likelihoods_R_A))

TypeError: 'NoneType' object is not subscriptable

In [10]:
drbart_model_R = DRBART(parser_dir = '../../../models/'+model_name+'/resource/',
                     strict_parser=False)
evaluator_R = conduct_evaluation.ConductEvaluation(drbart_model_R, SampleOutcomes_DRBART_Normal_R,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                    },
                                     test_event_log, n=N)
likelihoods_R = evaluator_R.sample_cases(False, True)

ValueError: invalid literal for int() with base 10: ''

In [11]:
np.mean([v.ln() for v in likelihoods_R[0].values()])

TypeError: 'NoneType' object is not subscriptable

In [12]:
np.mean(get_pscores(likelihoods_R))

TypeError: 'NoneType' object is not subscriptable

In [13]:
drbart_model_R_A_S = DRBART(parser_dir = '../../../models/'+model_name+'/concept-name_resource_seconds-in-day/',
                     strict_parser=False)
evaluator_R_A_S = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S, SampleOutcomes_DRBART_Normal_R_A_S,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S = evaluator_R_A_S.sample_cases(False, True)

FileNotFoundError: [Errno 2] No such file or directory: '../../../models/bpic_2017_all_2/concept-name_resource_seconds-in-day/ucuts.json'

In [14]:
np.mean([v.ln() for v in likelihoods_R_A_S[0].values()])

TypeError: 'NoneType' object is not subscriptable

In [15]:
np.mean(get_pscores(likelihoods_R_A_S))

TypeError: 'NoneType' object is not subscriptable

In [16]:
drbart_model_R_A_S_AC = DRBART(parser_dir = '../../../models/'+model_name+'/concept-name_resource_seconds-in-day_activity-count/',
                     strict_parser=False)
evaluator_R_A_S_AC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_AC, SampleOutcomes_DRBART_Normal_R_A_S_AC,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'known_activities' : known_activities,
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S_AC = evaluator_R_A_S_AC.sample_cases(False, True)

  0%|                                                                                                      | 0/6018 [00:00<?, ?it/s]

  0%|                                                                                         | 1/6018 [04:07<414:02:51, 247.73s/it]

  0%|                                                                                         | 2/6018 [10:05<521:46:25, 312.23s/it]

  0%|▍                                                                                         | 30/6018 [12:22<29:00:11, 17.44s/it]

  1%|▌                                                                                         | 37/6018 [14:21<28:46:16, 17.32s/it]

...

 99%|█████████████████████████████████████████████████████████████████████████████████████████▎| 5976/6018 [00:57<00:00, 104.08it/s]

 99%|█████████████████████████████████████████████████████████████████████████████████████████▌| 5987/6018 [00:57<00:00, 102.30it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████▋| 5998/6018 [00:57<00:00, 103.40it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████▊| 6009/6018 [00:57<00:00, 99.37it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 6018/6018 [00:57<00:00, 104.28it/s]

In [17]:
np.mean([v.ln() for v in likelihoods_R_A_S_AC[0].values()])

Decimal('-7.610956833016935304931266499')

In [18]:
np.mean(get_pscores(likelihoods_R_A_S_AC))

np.float64(227593640310.95712)

In [19]:
drbart_model_R_A_S_RC = DRBART(parser_dir = '../../../models/'+model_name+'/concept-name_resource_seconds-in-day_resource-count/',
                     strict_parser=False)
evaluator_R_A_S_RC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_RC, SampleOutcomes_DRBART_Normal_R_A_S_RC,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'known_resources' : known_resources
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S_RC = evaluator_R_A_S_RC.sample_cases(False, True)

  0%|                                                                                                      | 0/6018 [00:00<?, ?it/s]

  0%|                                                                                          | 1/6018 [01:13<122:15:00, 73.14s/it]

  0%|                                                                                         | 2/6018 [04:37<250:46:33, 150.07s/it]

  0%|▍                                                                                         | 30/6018 [05:57<14:45:52,  8.88s/it]

  1%|▊                                                                                          | 52/6018 [06:36<8:35:14,  5.18s/it]

...

 99%|█████████████████████████████████████████████████████████████████████████████████████████▎| 5968/6018 [00:51<00:00, 108.77it/s]

 99%|█████████████████████████████████████████████████████████████████████████████████████████▍| 5980/6018 [00:51<00:00, 110.50it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████▌| 5992/6018 [00:51<00:00, 112.93it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████▊| 6005/6018 [00:51<00:00, 110.71it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 6018/6018 [00:52<00:00, 115.58it/s]

In [20]:
np.mean([v.ln() for v in likelihoods_R_A_S_RC[0].values()])

Decimal('-7.061691532369009919623377607')

In [21]:
np.mean(get_pscores(likelihoods_R_A_S_RC))

np.float64(10168413778068.527)

In [22]:
drbart_model_R_A_S_RC_AC = DRBART(parser_dir = '../../../models/'+model_name+'/concept-name_resource_seconds-in-day_activity-count_resource_count/',
                     strict_parser=False)
evaluator_R_A_S_RC_AC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_RC_AC, SampleOutcomes_DRBART_Normal_R_A_S_RC_AC,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'known_resources' : known_resources,
                                                        'known_activities' : known_activities,
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S_RC_AC = evaluator_R_A_S_RC_AC.sample_cases(False, True)

  0%|                                                                                                      | 0/6018 [00:00<?, ?it/s]

  0%|                                                                                         | 1/6018 [02:07<212:25:38, 127.10s/it]

  0%|                                                                                         | 2/6018 [08:24<458:04:08, 274.11s/it]

  1%|▊                                                                                         | 52/6018 [09:47<13:15:49,  8.00s/it]

  1%|▊                                                                                         | 54/6018 [11:07<16:02:50,  9.69s/it]

...

 99%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊ | 5971/6018 [00:49<00:00, 127.78it/s]

 99%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏| 5984/6018 [00:49<00:00, 128.29it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍| 5997/6018 [00:49<00:00, 122.19it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 6010/6018 [00:49<00:00, 120.85it/s]

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6018/6018 [00:49<00:00, 121.20it/s]

In [23]:
np.mean([v.ln() for v in likelihoods_R_A_S_RC_AC[0].values()])

Decimal('-11.19300358207597152816208070')

In [24]:
np.mean(get_pscores(likelihoods_R_A_S_RC_AC))

np.float64(3516355130947.3247)

In [25]:
drbart_model_R_A_S_RC_AC_V = DRBART(parser_dir = '../../../models/'+model_name+'/concept-name_resource_seconds-in-day_activity-count_resource_count_amount/',
                     strict_parser=False)
evaluator_R_A_S_RC_AC_V = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_RC_AC_V, SampleOutcomes_DRBART_Normal_R_A_S_RC_AC_V,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'amount' : 'case.RequestedAmount_start',
                                                        'known_resources' : known_resources,
                                                        'known_activities' : known_activities,
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S_RC_AC_V = evaluator_R_A_S_RC_AC_V.sample_cases(False, True)

  0%|                                                                                                                                                                    | 0/6018 [00:00<?, ?it/s]

  0%|                                                                                                                                                                    | 0/6018 [00:00<?, ?it/s]

TypeError: '<' not supported between instances of 'NoneType' and 'float'

In [26]:
np.mean([v.ln() for v in likelihoods_R_A_S_RC_AC_V[0].values()])

TypeError: 'NoneType' object is not subscriptable

In [27]:
np.mean(get_pscores(likelihoods_R_A_S_RC_AC_V))

TypeError: 'NoneType' object is not subscriptable

In [28]:
drbart_model_R_A_S_D = DRBART(parser_dir = '../../../models/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week/',
                     strict_parser=False)
evaluator_R_A_S_D = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_D, SampleOutcomes_DRBART_Normal_R_A_S_D,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S_D = evaluator_R_A_S_D.sample_cases(False, True)

  0%|                                                                                                                                                                    | 0/6018 [00:00<?, ?it/s]

  0%|                                                                                                                                                       | 1/6018 [03:58<399:05:51, 238.78s/it]

  0%|                                                                                                                                                       | 2/6018 [10:37<556:09:56, 332.81s/it]

  0%|▍                                                                                                                                                       | 18/6018 [12:20<48:21:50, 29.02s/it]

  0%|▌                                                                                                                                                       | 21/6018 [12:25<39:17:08, 23.58s/it]

...

 99%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋ | 5966/6018 [00:52<00:00, 126.20it/s]

 99%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ | 5979/6018 [00:52<00:00, 126.42it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎| 5992/6018 [00:52<00:00, 126.03it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋| 6005/6018 [00:52<00:00, 124.29it/s]

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6018/6018 [00:52<00:00, 114.51it/s]

In [29]:
np.mean([v.ln() for v in likelihoods_R_A_S_D[0].values()])

Decimal('-4.255832272851605571343231609')

In [30]:
np.mean(get_pscores(likelihoods_R_A_S_D))

np.float64(855827.5124842082)

In [31]:
drbart_model_R_A_S_D_RC_AC = DRBART(parser_dir = '../../../models/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week_activity-count_resoure-count/',
                     strict_parser=False)
evaluator_R_A_S_D_RC_AC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_D_RC_AC, SampleOutcomes_DRBART_Normal_R_A_S_D_RC_AC,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'known_resources' : known_resources,
                                                        'known_activities' : known_activities,
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S_D_RC_AC = evaluator_R_A_S_D_RC_AC.sample_cases(False, True)

  0%|                                                                                                                                                                    | 0/6018 [00:00<?, ?it/s]

  0%|                                                                                                                                                        | 1/6018 [01:29<149:54:18, 89.69s/it]

  0%|                                                                                                                                                       | 2/6018 [03:33<183:48:42, 109.99s/it]

  0%|▌                                                                                                                                                       | 20/6018 [03:47<12:30:47,  7.51s/it]

  0%|▌                                                                                                                                                       | 24/6018 [04:33<14:04:13,  8.45s/it]

...

 99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊ | 5966/6018 [00:44<00:00, 126.25it/s]

 99%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏| 5979/6018 [00:45<00:00, 122.42it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍| 5994/6018 [00:45<00:00, 130.16it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 6008/6018 [00:45<00:00, 127.04it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6018/6018 [00:45<00:00, 132.66it/s]

In [32]:
np.mean([v.ln() for v in likelihoods_R_A_S_D_RC_AC[0].values()])

Decimal('-10.98639840832898627665170719')

In [33]:
np.mean(get_pscores(likelihoods_R_A_S_D_RC_AC))

np.float64(1067699436023.2373)

In [34]:
results = {
    'drbart_model_A' : likelihoods_A,
    'drbart_model_R' : likelihoods_R,
    'drbart_model_R_A' : likelihoods_R_A,
    'drbart_model_R_A_S' : likelihoods_R_A_S,
    'drbart_model_R_A_S_AC' : likelihoods_R_A_S_AC,
    'drbart_model_R_A_S_RC' : likelihoods_R_A_S_RC,
    'drbart_model_R_A_S_RC_AC' : likelihoods_R_A_S_RC_AC,
    'drbart_model_R_A_S_RC_AC_V' : likelihoods_R_A_S_RC_AC_V,
    'drbart_model_R_A_S_D' : likelihoods_R_A_S_D,
    'drbart_model_R_A_S_D_RC_CC' : likelihoods_R_A_S_D_RC_AC
}
with open('./'+model_name+'_dr_bart_evaluation_'+log_name+'.pickle', 'wb') as handle:
    pickle.dump(results, handle)